# Agent2Agent (A2A) — demo notebook

Companion to the deck **Agent2Agent (A2A) — protocol specification, interaction model, and the
implementation provided by Microsoft Foundry Agent Service**. Each step names the slide it
illustrates.

The notebook is in two parts.

| Part | Steps | Needs |
|---|---|---|
| **1 · The protocol** | 0–7 | Nothing but Python. A real A2A server runs inside this notebook. |
| **2 · Microsoft Foundry** | 8–11 | A Foundry project, an agent, and the roles listed in step 8. |

Part 1 is vendor-neutral and runs offline once the packages are installed — it is the part to
demonstrate live. Part 2 shows the Foundry configuration exactly as the documentation defines it;
those cells are guarded so the notebook runs end to end even without a project.

**Versions.** Protocol **A2A 1.0** throughout. Version 1.0 renamed methods, flattened the content
model and changed enum serialisation, so code written against 0.3 does not run here — step 7 shows
the difference on the wire.

**Docs**
- Specification — <https://a2a-protocol.org/latest/specification>
- Python SDK — <https://github.com/a2aproject/a2a-python>
- A2A tool in Foundry — <https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/tools/agent-to-agent>
- Inbound A2A endpoint — <https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/enable-agent-to-agent-endpoint>

## 0 · Install and import

`a2a-sdk[http-server]` brings Starlette and the SSE support the server needs. `uvicorn` is **not**
a dependency of the SDK, so install it explicitly.

Pin `a2a-sdk>=1.1.4`. Builds before 1.1.5 cap `protobuf<7`, which is fine with this repository's
protobuf 6.x; the dev container's package feed may not carry 1.1.5 yet, and pip takes it once it does.


In [2]:
%pip install -q "a2a-sdk[http-server]>=1.1.4" "uvicorn>=0.30" "httpx>=0.28.1"



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import asyncio, json, re, uuid, httpx, uvicorn
from starlette.applications import Starlette

# --- server side ---
from a2a.server.agent_execution.agent_executor import AgentExecutor
from a2a.server.agent_execution.context import RequestContext
from a2a.server.events.event_queue import EventQueue
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.routes import create_agent_card_routes, create_jsonrpc_routes
from a2a.server.tasks.inmemory_task_store import InMemoryTaskStore
from a2a.server.tasks.task_updater import TaskUpdater

# --- shared types (protobuf-generated in 1.0, not Pydantic) ---
from a2a.types import (
    AgentCapabilities, AgentCard, AgentInterface, AgentProvider, AgentSkill,
    CancelTaskRequest, GetTaskRequest, Message, Part, Role, SendMessageRequest,
    Task, TaskState, TaskStatus,
)

# --- client side ---
from a2a.client import A2ACardResolver, ClientConfig, create_client
from a2a.helpers import get_artifact_text, get_message_text
from a2a.utils.constants import AGENT_CARD_WELL_KNOWN_PATH, VERSION_HEADER, PROTOCOL_VERSION_CURRENT

HOST, PORT = "127.0.0.1", 41241
BASE       = f"http://{HOST}:{PORT}"
RPC_PATH   = "/a2a/jsonrpc"

print("agent card path :", AGENT_CARD_WELL_KNOWN_PATH)
print("version header  :", VERSION_HEADER, "=", PROTOCOL_VERSION_CURRENT)

agent card path : /.well-known/agent-card.json
version header  : A2A-Version = 1.0


## 1 · The Agent Card — the manifest a client reads first  *(slide 7)*

Everything a caller needs before its first request: identity, version, declared skills, capabilities,
and the interfaces it can be reached on.

Three things are worth watching here, because they are exactly what changed in 1.0:

- **`supported_interfaces`** replaces the 0.3 `url` + `preferred_transport` + `additional_interfaces`.
  It is an **ordered** list — a client takes the first binding it supports, so order expresses preference.
- **`protocol_version` sits on each interface**, not on the card, so one agent can serve several
  protocol versions at once. Set it to `"1.0"` explicitly: the resolver treats a missing version as 0.3
  and will silently hand you a compatibility client.
- **`capabilities`** declares what the agent supports. A caller that needs streaming must check
  `capabilities.streaming` rather than trying and handling the failure.

In [3]:
AGENT_CARD = AgentCard(
    name="Claims Triage Agent",
    description="Triages a motor insurance claim and returns a routing decision.",
    provider=AgentProvider(organization="Contoso Insurance", url="https://example.com"),
    version="1.0.0",
    capabilities=AgentCapabilities(streaming=True, push_notifications=False),
    default_input_modes=["text"],
    default_output_modes=["text"],
    skills=[
        AgentSkill(
            id="triage-claim",
            name="Triage a claim",
            description="Assesses a claim and returns a routing decision and a priority.",
            tags=["insurance", "triage"],
            examples=["Windscreen chip on policy MOT-4471", "Rear-end collision, policy MOT-9930"],
            input_modes=["text"], output_modes=["text"],
        )
    ],
    supported_interfaces=[
        AgentInterface(protocol_binding="JSONRPC", protocol_version="1.0", url=f"{BASE}{RPC_PATH}"),
    ],
)

print(AGENT_CARD.name, "·", AGENT_CARD.version)
print("skills          :", [s.id for s in AGENT_CARD.skills])
print("interfaces      :", [(i.protocol_binding, i.protocol_version) for i in AGENT_CARD.supported_interfaces])
print("streaming       :", AGENT_CARD.capabilities.streaming)

Claims Triage Agent · 1.0.0
skills          : ['triage-claim']
interfaces      : [('JSONRPC', '1.0')]
streaming       : True


## 2 · The remote agent — a task, its states, and an artifact  *(slides 6, 9)*

`AgentExecutor` is the one class you implement. It receives a `RequestContext` and an `EventQueue`,
and drives the task through the state machine on slide 9.

Version 1.0 **enforces** the streaming contract at runtime, so the order below is not stylistic:

1. enqueue a **`Task`** first, then
2. zero or more status or artifact updates, until
3. a terminal state.

Enqueue a status update before the task, or mix in a bare `Message`, and the SDK raises
`InvalidAgentResponseError`.

The agent deliberately does two different things, to show both halves of the lifecycle:

- **no policy number in the message** → `TASK_STATE_INPUT_REQUIRED`. The task *suspends*; it has not
  failed, and it keeps its id so the caller can answer on the same task.
- **policy number present** → work, emit the result as an **artifact**, then `TASK_STATE_COMPLETED`.

Note that the result is returned as an **artifact**, not as message text. The specification is
explicit about this: *"Messages SHOULD NOT be used to deliver task outputs. Results SHOULD BE
returned using Artifacts associated with a Task."*

In [4]:
POLICY = re.compile(r"\b([A-Z]{3}-\d{4})\b")

def triage(text: str) -> str:
    low = text.lower()
    if any(w in low for w in ("collision", "injury", "fire", "theft")):
        return "priority=HIGH route=adjuster-queue reason=bodily-risk-or-total-loss"
    if any(w in low for w in ("windscreen", "windshield", "glass", "chip")):
        return "priority=LOW route=glass-partner reason=fast-track-repair"
    return "priority=MEDIUM route=desk-assessor reason=default-path"


class ClaimsTriageExecutor(AgentExecutor):
    """Illustrative agent: suspends for input, then returns its result as an artifact."""

    async def execute(self, context: RequestContext, event_queue: EventQueue) -> None:
        user_message = context.message
        task_id, context_id = context.task_id, context.context_id
        if not user_message or not task_id or not context_id:
            return

        updater = TaskUpdater(event_queue=event_queue, task_id=task_id, context_id=context_id)

        # A NEW task must be announced before any update. A task we are resuming already exists.
        if context.current_task is None:
            await event_queue.enqueue_event(
                Task(id=task_id, context_id=context_id,
                     status=TaskStatus(state=TaskState.TASK_STATE_SUBMITTED),
                     history=[user_message])
            )

        await updater.start_work(
            message=updater.new_agent_message(parts=[Part(text="Reading the claim...")])
        )

        # Everything said on this task so far, so a resumed turn can see the original claim.
        said = " ".join(
            get_message_text(m) for m in (context.current_task.history if context.current_task else [])
        ) + " " + context.get_user_input()

        if not POLICY.search(said):
            # Interrupted, not failed. The task waits; the caller answers on the same task id.
            await updater.requires_input(
                message=updater.new_agent_message(parts=[Part(
                    text="Which policy is this claim against? Please give the policy number, e.g. MOT-4471."
                )])
            )
            return

        await asyncio.sleep(0.4)                      # stand-in for real work
        policy = POLICY.search(said).group(1)
        await updater.add_artifact(
            parts=[Part(text=f"policy={policy} {triage(said)}")],
            name="triage-decision", last_chunk=True,
        )
        await updater.complete()

    async def cancel(self, context: RequestContext, event_queue: EventQueue) -> None:
        await TaskUpdater(event_queue=event_queue,
                          task_id=context.task_id or "",
                          context_id=context.context_id or "").cancel()

print("executor ready")

executor ready


## 3 · Serve it — the routes an A2A server exposes  *(slide 8)*

Version 1.0 removed the `A2AStarletteApplication` wrapper. You now compose plain Starlette routes
from two factories: one for the well-known card, one for the JSON-RPC binding. The task store is
what gives the server its memory of a task between turns — swap `InMemoryTaskStore` for a database
store in production.

`uvicorn.Server.serve()` is a coroutine, so it runs as a task on the notebook's own event loop.

In [5]:
request_handler = DefaultRequestHandler(
    agent_executor=ClaimsTriageExecutor(),
    task_store=InMemoryTaskStore(),
    agent_card=AGENT_CARD,          # required in 1.0 — it used to live on the app wrapper
)

routes  = list(create_agent_card_routes(agent_card=AGENT_CARD))
routes += list(create_jsonrpc_routes(request_handler=request_handler, rpc_url=RPC_PATH))
app = Starlette(routes=routes)

for r in routes:
    print(f"{','.join(sorted(r.methods or [])):8s} {r.path}")

server = uvicorn.Server(uvicorn.Config(app, host=HOST, port=PORT, log_level="warning"))
server_task = asyncio.create_task(server.serve())
await asyncio.sleep(1.0)
print("\nserving on", BASE)

GET,HEAD /.well-known/agent-card.json
POST     /a2a/jsonrpc

serving on http://127.0.0.1:41241


## 4 · Discovery — fetch the card over HTTP  *(slide 7)*

The default mechanism: one `GET` to the well-known path. The other two the specification names are a
curated registry (for which it defines **no** API) and direct configuration.

The card comes back as ProtoJSON, so the wire form is **lowerCamelCase** even though the Python
attributes are snake_case.

In [6]:
async with httpx.AsyncClient(timeout=10) as hx:
    raw = (await hx.get(f"{BASE}{AGENT_CARD_WELL_KNOWN_PATH}")).json()

print(json.dumps(raw, indent=2)[:900], "...\n")

async with httpx.AsyncClient(timeout=10) as hx:
    resolved = await A2ACardResolver(hx, BASE).get_agent_card()

print("resolved      :", resolved.name, resolved.version)
print("first binding :", resolved.supported_interfaces[0].protocol_binding,
      resolved.supported_interfaces[0].protocol_version, "->",
      resolved.supported_interfaces[0].url)

{
  "name": "Claims Triage Agent",
  "description": "Triages a motor insurance claim and returns a routing decision.",
  "supportedInterfaces": [
    {
      "url": "http://127.0.0.1:41241/a2a/jsonrpc",
      "protocolBinding": "JSONRPC",
      "protocolVersion": "1.0"
    }
  ],
  "provider": {
    "url": "https://example.com",
    "organization": "Contoso Insurance"
  },
  "version": "1.0.0",
  "capabilities": {
    "streaming": true,
    "pushNotifications": false
  },
  "defaultInputModes": [
    "text"
  ],
  "defaultOutputModes": [
    "text"
  ],
  "skills": [
    {
      "id": "triage-claim",
      "name": "Triage a claim",
      "description": "Assesses a claim and returns a routing decision and a priority.",
      "tags": [
        "insurance",
        "triage"
      ],
      "examples": [
        "Windscreen chip on policy MOT-4471",
        "Rear-end collision, policy MOT-993 ...

resolved      : Claims Triage Agent 1.0.0
first binding : JSONRPC 1.0 -> http://127.0.0.1:4124

## 5 · Send a message and watch the task  *(slides 6, 9, 10)*

`send_message` is **always** an async iterator in 1.0 — there is no separate streaming call. If the
server or the client config does not support streaming, the SDK performs a unary call and wraps the
result, so one loop handles both modes.

Each item is a `StreamResponse` carrying exactly one of `task`, `message`, `status_update` or
`artifact_update`. Because these are protobuf messages, you test which one with `HasField()`, not
`isinstance`.

Watch the states go past: `TASK_STATE_SUBMITTED` → `TASK_STATE_WORKING` → `TASK_STATE_COMPLETED`,
with the result arriving as an artifact.

In [12]:
def new_text_message(text, *, task_id=None, context_id=None):
    """A user message. Note the flattened Part: no TextPart wrapper in 1.0."""
    return Message(
        role=Role.ROLE_USER,
        message_id=str(uuid.uuid4()),
        parts=[Part(text=text)],
        task_id=task_id or "",
        context_id=context_id or "",
    )


async def send(client, text, *, task_id=None, context_id=None):
    """Send one message; print every event; return (task_id, context_id, final_state)."""
    req = SendMessageRequest(message=new_text_message(text, task_id=task_id, context_id=context_id))
    t_id, c_id, state = task_id, context_id, None
    print(f"\n>>> {text}")
    async for ev in client.send_message(req):
        if ev.HasField("task"):
            t_id, c_id = ev.task.id, ev.task.context_id
            state = ev.task.status.state
            print(f"    task           {t_id[:8]}…  {TaskState.Name(state)}")
            # A unary (non-streaming) reply carries its results inside the final Task.
            for art in ev.task.artifacts:
                print(f"    artifact       {art.name}: {get_artifact_text(art, delimiter=' ')}")
            if ev.task.status.HasField("message"):
                print(f"    message        {get_message_text(ev.task.status.message, delimiter=' ')}")
        elif ev.HasField("status_update"):
            state = ev.status_update.status.state
            note = ""
            if ev.status_update.status.HasField("message"):
                note = "  \u2014 " + get_message_text(ev.status_update.status.message, delimiter=" ")
            print(f"    status         {TaskState.Name(state)}{note}")
        elif ev.HasField("artifact_update"):
            art = ev.artifact_update.artifact
            print(f"    artifact       {art.name}: {get_artifact_text(art, delimiter=' ')}")
        elif ev.HasField("message"):
            print(f"    message        {get_message_text(ev.message, delimiter=' ')}")
    return t_id, c_id, state


hx = httpx.AsyncClient(timeout=30)
client = await create_client(resolved, client_config=ClientConfig(httpx_client=hx))

task_id, context_id, state = await send(
    client, "Windscreen chip on the motorway, policy MOT-4471."
)
print("\nfinal:", TaskState.Name(state))



>>> Windscreen chip on the motorway, policy MOT-4471.
    task           852f7cf4…  TASK_STATE_SUBMITTED
    status         TASK_STATE_WORKING  — Reading the claim...
    artifact       triage-decision: policy=MOT-4471 priority=LOW route=glass-partner reason=fast-track-repair
    status         TASK_STATE_COMPLETED

final: TASK_STATE_COMPLETED


## 6 · Interrupted, not failed — and why the ids matter  *(slides 6, 9)*

This is the capability a tool call cannot express, and the reason the protocol exists.

The first message omits a policy number, so the agent moves to **`TASK_STATE_INPUT_REQUIRED`** and
stops. The task is not finished and has not failed — it is *suspended*. The caller answers by sending
a second message **with the same `task_id`**, and the agent resumes that task rather than starting a
new one.

Then the contrast: a **completed** task is terminal and can never restart. Refining the result means a
**new task** carrying the **same `context_id`** — which is why `context_id`, not `task_id`, is the
value to correlate on in your own logs.

In [10]:
t2, c2, s2 = await send(client, "Someone reversed into my rear bumper in a car park.")
assert s2 == TaskState.TASK_STATE_INPUT_REQUIRED          # interrupted, NOT a terminal state
print("   ->", TaskState.Name(s2), "· the task is waiting, not finished")

# Answer ON THE SAME TASK: pass task_id (and context_id) back.
t3, c3, s3 = await send(client, "It is policy MOT-9930.", task_id=t2, context_id=c2)
assert s3 == TaskState.TASK_STATE_COMPLETED
print("   ->", TaskState.Name(s3), "· same task id:", t3 == t2)


>>> Someone reversed into my rear bumper in a car park.
    task           8590dee6…  TASK_STATE_SUBMITTED
    status         TASK_STATE_WORKING  — Reading the claim...
    status         TASK_STATE_INPUT_REQUIRED  — Which policy is this claim against? Please give the policy number, e.g. MOT-4471.
   -> TASK_STATE_INPUT_REQUIRED · the task is waiting, not finished

>>> It is policy MOT-9930.
    status         TASK_STATE_WORKING  — Reading the claim...
    artifact       triage-decision: policy=MOT-9930 priority=MEDIUM route=desk-assessor reason=default-path
    status         TASK_STATE_COMPLETED
   -> TASK_STATE_COMPLETED · same task id: True


In [11]:
# A terminal task cannot be reopened. Refinement is a NEW task in the SAME context.
t4, c4, s4 = await send(client, "Re-check that claim, the driver reported neck pain.", context_id=c2)

print("\nsame context :", c4 == c2)
print("new task     :", t4 != t2)
print("final state  :", TaskState.Name(s4))


>>> Re-check that claim, the driver reported neck pain.
    task           258d5e7f…  TASK_STATE_SUBMITTED
    status         TASK_STATE_WORKING  — Reading the claim...
    status         TASK_STATE_INPUT_REQUIRED  — Which policy is this claim against? Please give the policy number, e.g. MOT-4471.

same context : True
new task     : True
final state  : TASK_STATE_INPUT_REQUIRED


## 7 · What it looks like on the wire  *(slides 8, 20)*

Two things to show a developer who has read older material.

**Method names.** Version 1.0 replaced the 0.3 `namespace/verb` form with the PascalCase operation
name — `message/send` became `SendMessage`. The SDK dispatches on the method name, so a version
mismatch produces a *wrong dispatch* rather than a clean error.

**The version header.** A request that does not state `A2A-Version` is interpreted as **0.3**. The
client library sets it for you; anything hand-rolled must not forget it. The second call below omits
it deliberately.

In [12]:
payload = {
    "jsonrpc": "2.0", "id": "demo-1", "method": "SendMessage",
    "params": {"message": {
        "role": "ROLE_USER", "messageId": str(uuid.uuid4()),
        "parts": [{"text": "Hail damage to the roof, policy MOT-2210."}],
    }},
}

async with httpx.AsyncClient(timeout=30) as raw_hx:
    r = await raw_hx.post(f"{BASE}{RPC_PATH}", json=payload,
                          headers={VERSION_HEADER: PROTOCOL_VERSION_CURRENT})
    print("with A2A-Version:", r.status_code)
    print(json.dumps(r.json(), indent=2)[:700], "...")

    payload["id"] = "demo-2"
    r2 = await raw_hx.post(f"{BASE}{RPC_PATH}", json=payload)   # header omitted -> read as 0.3
    body = r2.json()
    print("\nwithout header  :", r2.status_code,
          "->", "error: " + body["error"]["message"] if "error" in body else "accepted as 0.3")

with A2A-Version: 200
{
  "result": {
    "task": {
      "id": "fcbc1bf5-1cc8-4fbb-be80-47615347466c",
      "contextId": "1d17b27f-555c-4783-9b17-54f1be02f15a",
      "status": {
        "state": "TASK_STATE_COMPLETED",
        "timestamp": "2026-09-28T12:47:58.760295Z"
      },
      "artifacts": [
        {
          "artifactId": "4ce4f7bb-f253-4d6f-b46d-e23ee51b1611",
          "name": "triage-decision",
          "parts": [
            {
              "text": "policy=MOT-2210 priority=MEDIUM route=desk-assessor reason=default-path"
            }
          ]
        }
      ],
      "history": [
        {
          "messageId": "3ad68ccc-a318-4b52-a50f-782d45de13ef",
          "contextId": "1d17b27f-555c-47 ...

without header  : 200 -> error: A2A version '0.3' is not supported by this handler. Expected version '1.0'.


### Read a task back, and shut the server down

`GetTaskRequest` takes a bare `id`. The stored task carries its full history and its artifacts — this
is what a monitoring or audit path reads.

In [13]:
fetched = await client.get_task(GetTaskRequest(id=task_id))
print("task      :", fetched.id[:8], "…")
print("state     :", TaskState.Name(fetched.status.state))
print("history   :", len(fetched.history), "messages")
for a in fetched.artifacts:
    print("artifact  :", a.name, "->", get_artifact_text(a, delimiter=" "))

task      : 6177a0f4 …
state     : TASK_STATE_COMPLETED
history   : 2 messages
artifact  : triage-decision -> policy=MOT-4471 priority=LOW route=glass-partner reason=fast-track-repair


In [14]:
await client.close()
await hx.aclose()
server.should_exit = True
await server_task
print("server stopped")

server stopped


---

# Part 2 · A2A in Microsoft Foundry

Everything above is the open protocol. This part is the Microsoft Foundry implementation, and the two
directions carry different release status:

| Direction | What it is | Status |
|---|---|---|
| **Outbound** — Foundry agent calls a remote A2A agent | the `a2a` tool, protocol 1.0 | Generally available |
| **Outbound**, older | the `a2a_preview` tool, protocol 0.3 | Preview — existing integrations only |
| **Inbound** — a Foundry agent is exposed over A2A | `agent_endpoint.protocol_configuration.a2a` | Preview |

The cells below are guarded: without a project they print the configuration they *would* send, so the
notebook still runs top to bottom. Fill in the environment variables to execute them for real.

In [15]:
%pip install -q "azure-ai-projects>=2.5.0" azure-identity python-dotenv


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [19]:
import os
from dotenv import load_dotenv
load_dotenv()

PROJECT_ENDPOINT = os.getenv("FOUNDRY_PROJECT_ENDPOINT", "")   # https://<account>.services.ai.azure.com/api/projects/<project>
A2A_CONNECTION   = os.getenv("FOUNDRY_A2A_CONNECTION", "claims-triage-a2a")
AGENT_NAME       = os.getenv("FOUNDRY_AGENT_NAME", "concierge-agent")                 # calls out over A2A (step 8)
TARGET_AGENT     = os.getenv("FOUNDRY_A2A_TARGET_AGENT", "claims-triage-foundry")     # exposed over A2A (step 9)
MODEL_DEPLOYMENT = os.getenv("FOUNDRY_MODEL_NAME", "gpt-5-mini")
TARGET_A2A_URL   = f"{PROJECT_ENDPOINT or '{project_endpoint}'}/agents/{TARGET_AGENT}/endpoint/protocols/a2a"

LIVE = bool(PROJECT_ENDPOINT)
if LIVE:
    from azure.identity import DefaultAzureCredential
    from azure.ai.projects import AIProjectClient
    credential = DefaultAzureCredential()
    project = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)

print("live Foundry calls:", LIVE, "" if LIVE else "— set FOUNDRY_PROJECT_ENDPOINT to enable")
print("A2A target       :", TARGET_A2A_URL)


live Foundry calls: True 
A2A target       : https://cog-tb7tpjtuee4ji.services.ai.azure.com/api/projects/cog-tb7tpjtuee4ji-project/agents/claims-triage-foundry/endpoint/protocols/a2a


## 8 · Outbound — call a remote A2A agent from a Foundry agent  *(slide 12)*

Three artefacts, and the interesting decisions are all in the first one.

1. a **project connection** of kind `RemoteA2A`, holding the target endpoint, the audience and the
   credential;
2. a **tool definition** of type `a2a` at protocol version `1.0`, referencing that connection;
3. an **agent version** that carries the tool.

**The target must be reachable from Foundry.** Foundry fetches the remote agent card at run time;
So this demo delegates to a **Foundry agent** (`claims-triage-foundry`) that step 9 exposes over A2A — the documented Foundry-to-Foundry pattern:

| Connection property | Value |
|---|---|
| `category` | `RemoteA2A` |
| `target` | `{project_endpoint}/agents/claims-triage-foundry/endpoint/protocols/a2a` (no card path) |
| `authType` | `AgenticIdentityToken` — the calling agent authenticates with its own identity |
| `audience` | `https://ai.azure.com` |

The cell creates the connection with `azd ai connection create` (the Python SDK can only read
connections), the agent version with the SDK, and grants the calling agent's identity
**Foundry Agent Consumer** role with `az role assignment create` so it may call the target.
Step 10 runs the agent once the target exists.

**Use a new connection name when you change the target.** 

**Roles:** *Foundry Project Manager* for the connection, *Foundry User* for the agent, and permission to
create role assignments (Owner or User Access Administrator) on the project.

Docs: <https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/tools/agent-to-agent> ·
<https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/enable-agent-to-agent-endpoint#connect-from-another-foundry-agent>

In [21]:
TOOL_DEFINITION = {
    "type": "a2a",                                   # GA. "a2a_preview" targets protocol 0.3.
    "a2a_version": "1.0",
    "base_url": TARGET_A2A_URL,
    "project_connection_id": "<resolved from the connection>",
}
print(json.dumps(TOOL_DEFINITION, indent=2))

if LIVE:
    from azure.ai.projects.models import A2ATool, A2AProtocolVersion, PromptAgentDefinition

    # 1 · RemoteA2A connection to the Foundry agent exposed in step 9, with agent identity auth.
    !azd ai connection create {A2A_CONNECTION} --project-endpoint "{PROJECT_ENDPOINT}" --kind remote-a2a --target "{TARGET_A2A_URL}" --auth-type agentic-identity --audience "https://ai.azure.com" --force --no-prompt
    connection_id = project.connections.get(A2A_CONNECTION).id
    project_scope = connection_id.split("/connections/")[0]

    # 2-3 · The tool, and an agent version that carries it.
    agent = project.agents.create_version(
        agent_name=AGENT_NAME,
        definition=PromptAgentDefinition(
            model=MODEL_DEPLOYMENT,
            instructions=("You triage insurance enquiries. When a claim needs assessment, "
                          "delegate it to the claims agent through your A2A tool and report its decision."),
            tools=[A2ATool(a2a_version=A2AProtocolVersion.V1_0, project_connection_id=connection_id)],
        ),
    )
    print("agent version  :", agent.name, agent.version)

    # The calling agent's own identity must be allowed to call the target's A2A endpoint.
    principal = project.agents.get(AGENT_NAME).instance_identity.principal_id
    !az role assignment create --assignee-object-id {principal} --assignee-principal-type ServicePrincipal --role eed3b665-ab3a-47b6-8f48-c9382fb1dad6 --scope "{project_scope}" -o none
    print("role           : Foundry Agent Consumer ->", AGENT_NAME, "identity (allow a few minutes to propagate)")
else:
    print("\n(skipped — no project endpoint set)")

{
  "type": "a2a",
  "a2a_version": "1.0",
  "base_url": "https://cog-tb7tpjtuee4ji.services.ai.azure.com/api/projects/cog-tb7tpjtuee4ji-project/agents/claims-triage-foundry/endpoint/protocols/a2a",
  "project_connection_id": "<resolved from the connection>"
}
2026/09/29 09:32:33 connections: no active azd environment: rpc error: code = Unknown desc = no project exists; to create a new project, run `azd init`


Connection "claims-triage-a2a" created in project "cog-tb7tpjtuee4ji-project".
Update available: 1.34.0 -> 1.34.2 (]8;;https://github.com/Azure/azure-dev/releases/tag/azure-dev-cli_1.34.2Release notes]8;;)
To update, run `curl -fsSL https://aka.ms/install-azd.sh | bash`

If you installed azd with custom options, use the same options when updating. See ]8;;https://aka.ms/azd/upgrade/linuxhttps://aka.ms/azd/upgrade/linux]8;; for details.
agent version  : concierge-agent 3
role           : Foundry Agent Consumer -> concierge-agent identity (allow a few minutes to propagate)


## 9 · Inbound — expose a Foundry agent over A2A  *(slide 13)*

The cell creates `claims-triage-foundry` — a plain prompt agent with **no tools**, so it can answer on
its own — and one `PATCH` sets its agent card and turns the protocol on. Two details decide whether
consumers succeed:

- **`responses` must stay in the configuration.** Inbound A2A rides on the responses protocol, and
  the PATCH **replaces `protocol_configuration` wholesale** — omit `responses` or `Entra` and the
  endpoint loses them.
- **An unversioned request is served protocol 0.3 by default**, in line with the specification's rule
  that an absent version header means 0.3. Tell consumers to resolve the `agentCard/v1.0` path or to
  send the header explicitly.

Expose an agent that does not itself depend on an unreachable A2A target: if the exposed agent's own
tool fails, the inbound call surfaces as a bare JSON-RPC `InternalError`.

Inbound is in preview, it is not available in the portal, and it is Microsoft Entra ID only — no keys,
and no anonymous access to the agent card. Callers need the **Foundry Agent Consumer** role.

Docs: <https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/enable-agent-to-agent-endpoint>


In [10]:
INBOUND_PATCH = {
    "agent_card": {
        "description": "Triages a motor insurance claim and returns a routing decision.",
        "version": "1.0",
        "skills": [{
            "id": "triage-claim",
            "name": "Triage a claim",
            "description": "Assesses a claim and returns a routing decision and a priority.",
        }],
    },
    "agent_endpoint": {
        "protocol_configuration": {
            "responses": {},     # required — inbound A2A rides on the responses protocol
            "a2a": {},
        }
    },
}
print(f"PATCH {PROJECT_ENDPOINT or '{project_endpoint}'}/agents/{TARGET_AGENT}?api-version=v1")
print(json.dumps(INBOUND_PATCH, indent=2))

if LIVE:
    from azure.ai.projects.models import PromptAgentDefinition

    target = project.agents.create_version(
        agent_name=TARGET_AGENT,
        definition=PromptAgentDefinition(
            model=MODEL_DEPLOYMENT,
            instructions=(
                "You triage motor insurance claims. If no policy number (format ABC-1234) is given, ask for it. "
                "Otherwise reply on one line: policy=<number> priority=<HIGH|MEDIUM|LOW> route=<queue> reason=<short>. "
                "Collisions, injury, fire or theft are HIGH to adjuster-queue; glass damage is LOW to glass-partner; "
                "anything else is MEDIUM to desk-assessor."
            ),
        ),
    )
    print("\ntarget agent version:", target.name, target.version)

    token = credential.get_token("https://ai.azure.com/.default").token
    resp = httpx.patch(
        f"{PROJECT_ENDPOINT}/agents/{TARGET_AGENT}?api-version=v1",
        headers={"Authorization": f"Bearer {token}",
                 "Content-Type": "application/merge-patch+json"},
        json=INBOUND_PATCH, timeout=60,
    )
    print(resp.status_code, resp.text[:300])
    resp.raise_for_status()
    print("\ncard :", f"{TARGET_A2A_URL}/agentCard/v1.0")
    print("tasks:", TARGET_A2A_URL)
else:
    print("\n(skipped — no project endpoint set)")


PATCH https://cog-tb7tpjtuee4ji.services.ai.azure.com/api/projects/cog-tb7tpjtuee4ji-project/agents/claims-triage-foundry?api-version=v1
{
  "agent_card": {
    "description": "Triages a motor insurance claim and returns a routing decision.",
    "version": "1.0",
    "skills": [
      {
        "id": "triage-claim",
        "name": "Triage a claim",
        "description": "Assesses a claim and returns a routing decision and a priority."
      }
    ]
  },
  "agent_endpoint": {
    "protocol_configuration": {
      "responses": {},
      "a2a": {}
    }
  }
}

target agent version: claims-triage-foundry 1
200 {
  "object": "agent",
  "id": "claims-triage-foundry",
  "name": "claims-triage-foundry",
  "state": "enabled",
  "versions": {
    "latest": {
      "metadata": {},
      "object": "agent.version",
      "id": "claims-triage-foundry:1",
      "name": "claims-triage-foundry",
      "version": "1",


card : https://cog-tb7tpjtuee4ji.services.ai.azure.com/api/projects/cog-tb7tpjtue

## 10 · Call a Foundry agent as an A2A client  *(slides 13, 14)*

The same SDK from part 1, pointed at the agent step 9 exposed. Three differences from the local server:

- **a bearer token for `https://ai.azure.com/.default`** on every request — anonymous access is not
  supported, not even for the agent card;
- **an explicit card path**, `agentCard/v1.0`, so you get the 1.0 card rather than the 0.3 default;
- **`streaming=False`** — Foundry's A2A endpoint does not stream, so the client makes unary calls.

The second cell closes the loop: `concierge-agent` from step 8 delegates to the same agent through its
A2A tool, so one Foundry agent calls another over the protocol.


In [13]:
if LIVE:
    token = credential.get_token("https://ai.azure.com/.default").token

    async with httpx.AsyncClient(headers={"Authorization": f"Bearer {token}"}, timeout=120) as fhx:
        f_card = await A2ACardResolver(fhx, TARGET_A2A_URL,
                                       agent_card_path="agentCard/v1.0").get_agent_card()
        print("resolved:", f_card.name, "·", f_card.version)

        f_client = await create_client(f_card, client_config=ClientConfig(streaming=False, httpx_client=fhx))
        await send(f_client, "Windscreen chip on the motorway, policy MOT-4471.")
        await f_client.close()
else:
    print("(skipped — no project endpoint set)")
    print("card :  {project_endpoint}/agents/{agent}/endpoint/protocols/a2a/agentCard/v1.0")
    print("token:  scope https://ai.azure.com/.default · role Foundry Agent Consumer")


resolved: claims-triage-foundry · 1.0

>>> Windscreen chip on the motorway, policy MOT-4471.
    task           resp_0d0…  TASK_STATE_COMPLETED
    artifact       : policy=MOT-4471 priority=LOW route=glass-partner reason=windscreen chip


In [23]:
if LIVE:
    # Foundry -> Foundry over A2A: concierge-agent (step 8) delegates to claims-triage-foundry (step 9).
    reply = project.get_openai_client().responses.create(
        input="Someone reversed into my rear bumper in a car park, policy MOT-9930. How is this routed?",
        extra_body={"agent_reference": {"name": AGENT_NAME, "type": "agent_reference"}},
    )
    print("status :", reply.status)
    print("items  :", [item.type for item in reply.output])
    print("\n" + reply.output_text)
else:
    print("(skipped — no project endpoint set)")


status : completed
items  : ['a2a_preview_call', 'a2a_preview_call_output', 'message']

Your claim under policy **MOT-9930** is routed to the **adjuster queue** with **high priority**.

**Reason:** Collision damage to your rear bumper.


## 11 · What to check before you design around this  *(slides 14, 15)*

**Identity.** Five outbound authentication methods, and only one of them preserves who the end user
is. If the remote agent enforces per-user authorisation, OAuth identity passthrough is the only
option that works — prove it early rather than at security review.

| Method | End-user context |
|---|---|
| Key-based | not preserved |
| Microsoft Entra ID — agent identity | not preserved |
| Microsoft Entra ID — project managed identity | not preserved |
| **OAuth identity passthrough** | **preserved** |
| Unauthenticated | not preserved |

**Constraints that rule designs out.** These are the ones to check before promising anything.

In [22]:
CONSTRAINTS = [
    ("Transport",      "protocol 1.0 is JSON-RPC only; 0.3 also allowed HTTP+JSON"),
    ("Modality",       "text only — file and structured data parts are not carried"),
    ("Streaming",      "server-sent events are not supported for A2A targets"),
    ("Task retention", "60 days from the most recent write; each write resets the window"),
    ("Tools per agent","128 registered tools; no A2A-specific quota published"),
    ("Timeout",        "not documented — measure it in your own environment"),
]
w = max(len(k) for k, _ in CONSTRAINTS)
for k, v in CONSTRAINTS:
    print(f"{k:<{w}}  {v}")

print("\nStatus, verified 17 September 2026")
print("  outbound a2a tool ....... generally available (protocol 1.0)")
print("  outbound a2a_preview .... preview (protocol 0.3, existing integrations)")
print("  inbound A2A endpoint .... preview")

Transport        protocol 1.0 is JSON-RPC only; 0.3 also allowed HTTP+JSON
Modality         text only — file and structured data parts are not carried
Streaming        server-sent events are not supported for A2A targets
Task retention   60 days from the most recent write; each write resets the window
Tools per agent  128 registered tools; no A2A-specific quota published
Timeout          not documented — measure it in your own environment

Status, verified 17 September 2026
  outbound a2a tool ....... generally available (protocol 1.0)
  outbound a2a_preview .... preview (protocol 0.3, existing integrations)
  inbound A2A endpoint .... preview


---

### Where to go next

| | |
|---|---|
| Specification | <https://a2a-protocol.org/latest/specification> |
| Python SDK and samples | <https://github.com/a2aproject/a2a-python> |
| A2A tool in Foundry | <https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/tools/agent-to-agent> |
| Inbound A2A endpoint | <https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/enable-agent-to-agent-endpoint> |
| A2A authentication | <https://learn.microsoft.com/en-us/azure/foundry/agents/concepts/agent-to-agent-authentication> |
| Delegation sample | <https://github.com/microsoft-foundry/foundry-samples> |

**Presenter note.** Part 1 is the live demo — steps 5 and 6 carry the argument, because the
input-required turn is the thing a tool call cannot express. Part 2 is configuration: show the two
JSON payloads and the identity table, and do not attempt a live publish in front of a customer.

**Presenter note.** Status and limits change on Microsoft's release cadence. Re-check the A2A tool
page and the migrate page before presenting; the delegation sample in `foundry-samples` still creates
the preview `a2a_preview` tool at protocol 0.3.